In [ ]:
#!/usr/bin/env python3
# ============================================================
# Con φ  ~  WASE  (Without Any Survey Estimate)
# ============================================================
#
# VERSION BLOCK
# -------------
# Set these two variables once here. All paths and output
# filenames are derived from them automatically.
# If you are running this notebook standalone (not via the
# master runner), uncomment the lines below and set your own
# BASE_DIR.
#
VERSION = "v1"
BASE_DIR_DEFAULT = "/content/drive/MyDrive/conphi"
#
# ── Per-script override (uncomment if running standalone) ────
# from pathlib import Path
# VERSION  = "v1"
# BASE_DIR = Path("/content/drive/MyDrive/conphi")   # 👈 change to your path
# ─────────────────────────────────────────────────────────────
#
# If BASE_DIR is not already defined (i.e. running standalone
# without the master runner), fall back to the default above.
try:
    BASE_DIR
except NameError:
    from pathlib import Path
    BASE_DIR = Path(BASE_DIR_DEFAULT)

BASE_DIR = Path(BASE_DIR)

# ============================================================
# Con φ WASE — Without Any Survey Estimate
# ============================================================
"""
Con φ WASE — Without Any Survey Estimate
==========================================

WASE is one of two sub-models in the Con φ system used by the
World Food Programme for global expenditure monitoring.  It is
used when a country has NO usable consumption survey, or when
a structural prediction is preferred over a survey-anchored
projection.  WASE predicts the full household consumption
distribution purely from country-level structural indicators.

CORE IDEA
---------
If we know a country's income level and structural
characteristics, we can estimate where its consumption
distribution sits.  WASE learns this mapping from countries
that do have surveys, then applies it to countries that do not.
This makes it the "never-seen countries" arm of Con φ.

MODEL SPECIFICATION
-------------------
Each observation is a single percentile from a single
country-year survey.  WASE predicts log-consumption at
percentile p as:

    log_cons = LEVEL + SHAPE × logit(p) + SPLINE(logit(p))

Where:
    LEVEL  — average consumption, driven by GDP per capita,
              mortality, education, fiscal indicators, plus
              region and subregion random effects
    SHAPE  — inequality (the log-logistic shape parameter
              gamma), varying by region and subregion
    SPLINE — nonlinear corrections via radial basis functions,
              capturing departures from the log-logistic shape
              (e.g. heavier tails, subsistence floors)

The logit(p) transformation reflects the log-logistic
(Fisk) distribution: if consumption is Fisk-distributed,
log_cons is linear in logit(p). The spline term relaxes
this assumption for the real-world distributions we observe.

HIERARCHICAL STRUCTURE
----------------------
The model uses a three-level Bayesian hierarchy:

    Global → Region (WB regions) → Subregion (region23)

This gives partial pooling: data-sparse subregions borrow
strength from their parent region and the global mean, while
data-rich subregions can deviate as the data warrants.  Both
the level (intercept) and shape (inequality) components have
their own hierarchy.

EMPIRICAL BAYES PRIORS
----------------------
Before fitting the full Bayesian model, a weighted OLS is run
on the training data to obtain sensible starting points for
the regression coefficients.  These OLS estimates become the
prior means for the Bayesian model, with pre-specified prior
SDs controlling how far the posterior can deviate.  This is
fold-specific — no data leakage.

SURVEY-COUNT WEIGHTING
-----------------------
Countries with many surveys would otherwise dominate the
training set.  WASE downweights them using 1/sqrt(n_surveys),
normalised to mean 1.  This ensures fragile states with one
or two surveys get proportional influence — exactly the
countries the model most needs to work for.  The weighting
is applied in covariate centering, the empirical Bayes OLS,
and the likelihood.

RBF PERCENTILE BASIS
---------------------
Five Gaussian radial basis functions placed at evenly spaced
points on the logit(p) axis capture nonlinear departures from
the log-logistic shape.  Each has a global coefficient plus a
region-level deviation, allowing different regions to have
different tail shapes.

JAX COMPILATION STRATEGY
--------------------------
All training arrays are padded to a fixed maximum size (PAD_N)
with a boolean obs_mask.  This means JAX compiles the SVI step
function exactly once across all LOCO folds, rather than
recompiling for each fold's different training set size.

ESTIMATION
----------
Stochastic Variational Inference (SVI) with a Block Neural
Autoregressive Flow (BNAF) guide, implemented in NumPyro/JAX,
with a cosine decay learning rate schedule.  SVI variance
inflation corrects for the known tendency of KL(q||p)
minimisation to underestimate posterior variance.

VALIDATION
----------
Two cross-validation phases:
    Phase 1 — Rolling LOCO: for each focal year, train on all
              data before that year, hold out one country at a
              time and predict its distribution.
    Phase 2 — All-time LOCO: train on all years, hold out one
              country entirely (trust score assessment).
    Phase 3 — Production: train on all data, predict every
              LMIC country-year in the target range.

INPUTS
------
    conphi/data - model inputs/conphi_v1_features_{year}.parquet

OUTPUTS
-------
    conphi/outputs/conphi_v1_wase_{mode_suffix}/
      ├── loco_rolling/
      │   └── {year}/
      │       ├── conphi_v1_wase_pred_{year}_{iso}_{sfx}.parquet
      │       └── conphi_v1_wase_metrics_{year}_{iso}_{sfx}.parquet
      ├── loco_alltime/
      │   ├── conphi_v1_wase_pred_alltime_{iso}_{sfx}.parquet
      │   └── conphi_v1_wase_metrics_alltime_{iso}_{sfx}.parquet
      ├── production/
      │   ├── conphi_v1_wase_production_predictions_{sfx}.parquet
      │   ├── conphi_v1_wase_production_params_{sfx}.parquet
      │   └── conphi_v1_wase_production_metadata_{sfx}.json
      └── master/
          ├── conphi_v1_wase_rolling_predictions_master_{sfx}.parquet
          ├── conphi_v1_wase_rolling_metrics_master_{sfx}.parquet
          ├── conphi_v1_wase_alltime_predictions_master_{sfx}.parquet
          ├── conphi_v1_wase_alltime_metrics_master_{sfx}.parquet
          ├── conphi_v1_wase_combined_metrics_master_{sfx}.parquet
          ├── conphi_v1_wase_run_metadata_{sfx}.json
          └── diagnostics/
"""

# ============================================================
# BLOCK 1 — IMPORTS & GPU SETUP
# ============================================================
!pip install numpyro arviz pyarrow optax

import os, random, sys, json, time, gc, warnings, platform, shutil
from pathlib import Path
import numpy as np, torch

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── GPU diagnostics ──────────────────────────────────────────
print("=" * 80)
print("CUDA / GPU DIAGNOSTICS")
print("=" * 80)
print(f"torch version:   {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not available. This model needs a GPU to run "
        "in reasonable time. In Colab: Runtime → Change runtime type → GPU."
    )
DEVICE = torch.device("cuda")
print(f"Using device:    {DEVICE}")
print(f"GPU name:        {torch.cuda.get_device_name(0)}")
print(f"GPU memory (GB): {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}")

torch.set_default_dtype(torch.float32)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

import jax
import jax.numpy as jnp
import jax.nn as jnn
import pandas as pd
import optax
import numpyro
import numpyro.distributions as dist
from scipy.special import logit as sp_logit
from jax import random as jrandom
from numpyro.infer import SVI, Trace_ELBO, Predictive
from numpyro.infer.autoguide import AutoBNAFNormal

warnings.filterwarnings("ignore")

print(f"JAX devices: {jax.devices()}")
USE_GPU = jax.default_backend() == "gpu"
numpyro.set_platform("cuda" if USE_GPU else "cpu")
numpyro.set_host_device_count(1)


# ============================================================
# BLOCK 2 — CONFIGURATION
# ============================================================
MODEL_NAME = "conphi_v1_wase"

# ── I/O directories ───────────────────────────────────────────
INPUT_DIR    = BASE_DIR / "data - model inputs"
INPUT_PREFIX = f"conphi_{VERSION}_features"

# ── Run mode ──────────────────────────────────────────────────
# "loco"       — Phases 1 & 2 only (rolling + all-time LOCO).
# "production" — Phase 3 only (train on all data, predict everywhere).
# "both"       — All three phases.
RUN_MODE = "both"
assert RUN_MODE in ("loco", "production", "both")

# ── Prediction mode ──────────────────────────────────────────
# "nowcast"  — predict the vintage year itself (horizon 0)
# "forecast" — predict FORECAST_HORIZON years ahead
PREDICTION_MODE  = "forecast"
FORECAST_HORIZON = 1
assert PREDICTION_MODE in ("nowcast", "forecast")
assert 1 <= FORECAST_HORIZON <= 5
MODE_SUFFIX = (
    "nowcast" if PREDICTION_MODE == "nowcast"
    else f"forecast_{FORECAST_HORIZON}yr"
)

# ── Output directories ────────────────────────────────────────
WASE_OUT_DIR   = BASE_DIR / "outputs" / f"{MODEL_NAME}_{MODE_SUFFIX}"
ROLLING_DIR    = WASE_OUT_DIR / "loco_rolling"
ALL_TIME_DIR   = WASE_OUT_DIR / "loco_alltime"
MASTER_DIR     = WASE_OUT_DIR / "master"
PRODUCTION_DIR = WASE_OUT_DIR / "production"

# Flush and recreate directories relevant to the current run mode
_dirs_to_flush = [MASTER_DIR]
if RUN_MODE in ("loco", "both"):
    _dirs_to_flush.extend([ROLLING_DIR, ALL_TIME_DIR])
if RUN_MODE in ("production", "both"):
    _dirs_to_flush.append(PRODUCTION_DIR)

for d in _dirs_to_flush:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
    print(f"  Flushed: {d}")

for d in [ROLLING_DIR, ALL_TIME_DIR, PRODUCTION_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print()

# ── Cross-validation setup ───────────────────────────────────
FOCAL_YEARS      = list(range(2015, 2027))
ALL_TIME_VINTAGE = 2025

# ── Production prediction settings ───────────────────────────
# Phase 3 trains on ALL data and predicts on every LMIC
# country-year in PRODUCTION_YEARS where covariates are
# available, whether or not a survey exists.
PRODUCTION_VINTAGE = ALL_TIME_VINTAGE
PRODUCTION_YEARS   = list(range(2015, 2027))
LMIC_INCOME_GROUPS = {"Low income", "Lower middle income", "Upper middle income"}

# ── Target variable ──────────────────────────────────────────
TARGET = "log_cons"

# When False, validation folds only evaluate rows that have
# observed outcomes. Phase 3 always predicts everywhere.
ALLOW_PURE_PREDICTION = False

# ── Minimum training set thresholds ──────────────────────────
MIN_TRAIN_ROWS      = 100
MIN_TRAIN_COUNTRIES = 5

# ── SVI hyperparameters ───────────────────────────────────────
SVI_STEPS               = 12_000
SVI_LR                  = 0.005
POSTERIOR_SAMPLES       = 1_000
POSTERIOR_SAMPLE_GROUPS = 4
# SVI underestimates posterior variance (a known property of
# KL(q||p) minimisation). We inflate observation noise post-hoc
# to achieve well-calibrated 90% predictive intervals.
SVI_VARIANCE_INFLATION  = 1.5

# ── Likelihood ────────────────────────────────────────────────
# Laplace is more robust to outliers than Gaussian — appropriate
# for consumption data with measurement error and extreme values.
USE_LAPLACE     = True
LIKELIHOOD_NAME = "Laplace" if USE_LAPLACE else "Normal"

# ── Covariate column names ────────────────────────────────────
GDP_COL        = "log_gdp_pp"
U5_COL         = "u5mort_lag3"
RURAL_COL      = "rural_pct_lag3"
EDU_FEM_COL    = "edu_mean_fem_lag3"
GDP_GROWTH_COL = "gdp_growth"
GOV_REV_COL    = "gov_rev_lag3"
RES_RENTS_COL  = "res_rents_lag3"

# ── Shape parameter bounds ────────────────────────────────────
SHAPE_MAX = 0.95
SHAPE_EPS = 1e-4

# ── Prior standard deviations for core covariates ────────────
PRIOR_SD = {
    "beta_gdp":      0.10,
    "beta_u5m":      0.10,
    "beta_rural":    0.003,
    "beta_edu_fem":  0.05,
}
GAMMA0_RAW_PRIOR_SD = 0.35
BETA0_PRIOR_SD      = 1.0

# ── Exploratory covariate priors (zero-centred, tight) ───────
EXTRA_PRIORS = {
    "beta_gdp_sq":          (0.0, 0.05),
    "beta_gdp_growth":      (0.0, 0.05),
    "beta_gdp_growth_abs":  (0.0, 0.05),
    "beta_gov_rev":         (0.0, 0.004),
    "beta_res_rents":       (0.0, 0.004),
}

# ── Hierarchical random effect prior SDs ─────────────────────
SIGMA_REGION_LEVEL_PRIOR  = 0.40
SIGMA_R23_LEVEL_PRIOR     = 0.20
SIGMA_REGION_SHAPE_PRIOR  = 0.12
SIGMA_R23_SHAPE_PRIOR     = 0.08

# ── RBF percentile basis ──────────────────────────────────────
# Five Gaussian bumps on the logit(p) axis capture nonlinear
# departures from the log-logistic shape at different parts
# of the distribution.
RBF_CENTERS          = np.array([-3.0, -1.5, 0.0, 1.5, 3.0], dtype=np.float32)
RBF_WIDTH            = 1.5
N_RBF_BASIS          = len(RBF_CENTERS)
ALPHA_SPLINE_PRIOR_SD      = 0.04
SIGMA_REGION_SPLINE_PRIOR  = 0.02

# ── Region string constants ───────────────────────────────────
REGION_ECA = "Europe & Central Asia"
REGION_LAC = "Latin America & Caribbean"
REGION_SSA = "Sub-Saharan Africa"

# ── Numerical safety ──────────────────────────────────────────
LOG_CONS_CLIP = (-20.0, 20.0)

# ── JAX cache management ──────────────────────────────────────
JAX_CLEAR_EVERY = 10

# PAD_N is set in __main__ after scanning the data files.
PAD_N = None


# ============================================================
# BLOCK 3 — HELPER FUNCTIONS
# ============================================================

def shape_transform(raw):
    """Map unconstrained real → (SHAPE_EPS, SHAPE_MAX) via sigmoid."""
    return SHAPE_EPS + (SHAPE_MAX - SHAPE_EPS) * jnn.sigmoid(raw)


def inverse_shape_transform(gamma):
    """Inverse of shape_transform: (SHAPE_EPS, SHAPE_MAX) → real."""
    gamma_scaled = np.clip(
        (gamma - SHAPE_EPS) / (SHAPE_MAX - SHAPE_EPS), 1e-6, 1 - 1e-6
    )
    return float(np.log(gamma_scaled / (1 - gamma_scaled)))


def safe_exp(x):
    """Exponentiate with clipping to avoid inf/NaN."""
    return np.exp(np.clip(x, *LOG_CONS_CLIP))


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)


def compute_rbf_basis(logit_p_values):
    """
    Compute Gaussian RBF basis matrix of shape (N, K).

    Each column is a Gaussian bump centred at one of the
    RBF_CENTERS, evaluated at each logit(p) value.  Together
    they form a smooth, localised basis for nonlinear shape
    corrections at different parts of the distribution.
    """
    lp      = np.asarray(logit_p_values, dtype=np.float32)[:, np.newaxis]
    centers = RBF_CENTERS[np.newaxis, :]
    return np.exp(-(lp - centers) ** 2 / (2 * RBF_WIDTH ** 2)).astype(np.float32)


def build_run_metadata():
    return {
        "model_name":            MODEL_NAME,
        "model_version":         VERSION,
        "likelihood":            LIKELIHOOD_NAME,
        "input_prefix":          INPUT_PREFIX,
        "prediction_mode":       PREDICTION_MODE,
        "forecast_horizon":      FORECAST_HORIZON if PREDICTION_MODE == "forecast" else 0,
        "mode_suffix":           MODE_SUFFIX,
        "target":                TARGET,
        "seed":                  SEED,
        "svi_steps":             SVI_STEPS,
        "svi_lr":                SVI_LR,
        "svi_variance_inflation": SVI_VARIANCE_INFLATION,
        "posterior_samples":     POSTERIOR_SAMPLES * POSTERIOR_SAMPLE_GROUPS,
        "focal_years":           FOCAL_YEARS,
        "pad_n":                 PAD_N,
        "extra_priors":          {k: {"mean": v[0], "sd": v[1]} for k, v in EXTRA_PRIORS.items()},
        "features":              [
            "survey_count_weighting", "rbf_percentile_basis",
            "weighted_centering", "obs_padding",
        ],
        "run_mode":              RUN_MODE,
        "production": {
            "vintage":        PRODUCTION_VINTAGE,
            "target_years":   PRODUCTION_YEARS,
            "lmic_income_groups": sorted(LMIC_INCOME_GROUPS),
        },
        "rbf_config": {
            "centers":    RBF_CENTERS.tolist(),
            "width":      RBF_WIDTH,
            "alpha_sd":   ALPHA_SPLINE_PRIOR_SD,
            "region_sd":  SIGMA_REGION_SPLINE_PRIOR,
        },
        "jax_backend": jax.default_backend(),
    }


# ============================================================
# BLOCK 4 — DATA PREPARATION: TEST GRID
# ============================================================

def prep_loco_test_grid(df_test_raw):
    """
    Prepare the test set for a LOCO fold.

    For country-years with survey data, retain the observed
    percentiles.  For country-years WITHOUT survey data, expand
    each row to the full 99-percentile grid so WASE can
    predict the entire distribution.
    """
    df_clean = df_test_raw.copy()
    if len(df_clean) == 0:
        return df_clean

    has_survey = df_clean[df_clean["percentile"].notna()].copy()
    no_survey  = df_clean[df_clean["percentile"].isna()].copy()

    if len(no_survey) > 0:
        no_survey = no_survey.drop(
            columns=["percentile", "p", "logit_p", "cons", TARGET, "pop"],
            errors="ignore",
        )
        pcts          = pd.DataFrame({"percentile": np.arange(1, 100)})
        pcts["p"]      = pcts["percentile"] / 100.0
        pcts["logit_p"] = sp_logit(pcts["p"].values)
        exploded = no_survey.merge(pcts, how="cross")
        df_final = pd.concat([has_survey, exploded], ignore_index=True)
    else:
        df_final = has_survey

    return df_final.sort_values(["year", "percentile"]).reset_index(drop=True)


# ============================================================
# BLOCK 5 — EVALUATION
# ============================================================

def evaluate_predictions(df_pred, validation_mode, focal_year, holdout_iso, horizon=0):
    """
    Compute prediction quality metrics for a single LOCO fold.

    Reports metrics on both the log scale (where WASE operates)
    and the consumption scale.  Coverage metrics indicate whether
    predictive intervals are well-calibrated.
    """
    df_eval = df_pred.dropna(subset=[TARGET]).copy()

    out = {
        "model_name":          MODEL_NAME,
        "likelihood":          LIKELIHOOD_NAME,
        "prediction_mode":     PREDICTION_MODE,
        "mode_suffix":         MODE_SUFFIX,
        "validation_mode":     validation_mode,
        "focal_year":          focal_year,
        "holdout_iso":         holdout_iso,
        "horizon":             int(horizon),
        "n_rows_saved":        int(len(df_pred)),
        "n_eval_rows":         int(len(df_eval)),
        "n_unseen_r23_rows":   int(df_pred["unseen_r23"].sum())    if "unseen_r23"    in df_pred.columns else 0,
        "n_unseen_region_rows": int(df_pred["unseen_region"].sum()) if "unseen_region" in df_pred.columns else 0,
        "mae_log":             np.nan,
        "rmse_log":            np.nan,
        "bias_log":            np.nan,
        "r2_log":              np.nan,
        "mape_pct_from_log":   np.nan,
        "mae_cons_from_mu":    np.nan,
        "rmse_cons_from_mu":   np.nan,
        "bias_cons_from_mu":   np.nan,
        "coverage_90_mu_pct":  np.nan,
        "coverage_90_y_pct":   np.nan,
    }

    if len(df_eval) == 0:
        return pd.DataFrame([out])

    y_true  = df_eval[TARGET].values
    y_hat   = df_eval["mu_mean"].values
    resid   = y_hat - y_true
    ss_res  = np.sum(resid ** 2)
    ss_tot  = np.sum((y_true - y_true.mean()) ** 2)

    cons_true = safe_exp(y_true)
    cons_hat  = safe_exp(y_hat)

    cov_mu = np.mean(
        (y_true >= df_eval["mu_q05"].values) & (y_true <= df_eval["mu_q95"].values)
    ) * 100
    cov_y  = np.mean(
        (y_true >= df_eval["y_q05"].values)  & (y_true <= df_eval["y_q95"].values)
    ) * 100

    out.update({
        "mae_log":            float(np.mean(np.abs(resid))),
        "rmse_log":           float(np.sqrt(np.mean(resid ** 2))),
        "bias_log":           float(resid.mean()),
        "r2_log":             float(1.0 - ss_res / ss_tot) if ss_tot > 1e-8 else np.nan,
        "mape_pct_from_log":  float(np.mean(np.abs(np.exp(np.clip(resid, -20, 20)) - 1.0)) * 100),
        "mae_cons_from_mu":   float(np.mean(np.abs(cons_hat - cons_true))),
        "rmse_cons_from_mu":  float(np.sqrt(np.mean((cons_hat - cons_true) ** 2))),
        "bias_cons_from_mu":  float((cons_hat - cons_true).mean()),
        "coverage_90_mu_pct": float(cov_mu),
        "coverage_90_y_pct":  float(cov_y),
    })
    return pd.DataFrame([out])


# ============================================================
# BLOCK 6 — POSTERIOR SUMMARIES
# ============================================================

def summarize_posterior(posterior_fit, info):
    """Extract posterior summary statistics for all model parameters."""
    samples = posterior_fit.get_samples()

    scalar_params = [
        ("beta0",               "intercept"),
        ("beta_gdp",            "gdp_elasticity"),
        ("beta_gdp_sq",         "gdp_curvature"),
        ("beta_u5m",            "u5_mortality_effect"),
        ("beta_rural",          "rural_share_effect"),
        ("beta_edu_fem",        "female_education_effect"),
        ("beta_gdp_growth",     "gdp_growth_effect"),
        ("beta_gdp_growth_abs", "growth_volatility_effect"),
        ("beta_gov_rev",        "gov_rev_level_effect"),
        ("beta_res_rents",      "res_rents_level_effect"),
        ("gamma0",              "baseline_inequality"),
        ("sigma_base",          "noise_baseline"),
        ("sigma_tail",          "noise_tail_widening"),
        ("sigma_region_level",  "region_level_pooling"),
        ("sigma_r23_level",     "subregion_level_pooling"),
        ("sigma_region_shape",  "region_shape_pooling"),
        ("sigma_r23_shape",     "subregion_shape_pooling"),
        ("sigma_region_spline", "region_spline_pooling"),
    ]

    rows = []
    for sn, dn in scalar_params:
        if sn not in samples:
            continue
        vals = np.array(samples[sn]).flatten()
        rows.append({
            "model_name":       MODEL_NAME,
            "prediction_mode":  PREDICTION_MODE,
            "mode_suffix":      MODE_SUFFIX,
            "validation_mode":  info["validation_mode"],
            "focal_year":       info["focal_year"],
            "holdout_iso":      info["holdout_iso"],
            "parameter":        dn,
            "posterior_mean":   float(np.mean(vals)),
            "posterior_q05":    float(np.percentile(vals, 5)),
            "posterior_q95":    float(np.percentile(vals, 95)),
            "posterior_sd":     float(np.std(vals)),
            "n_train_rows":     info["n_train_rows"],
            "n_train_countries": info["n_train_countries"],
        })

    if "alpha_spline" in samples:
        for k in range(N_RBF_BASIS):
            vals = np.array(samples["alpha_spline"])[:, k]
            rows.append({
                "model_name":       MODEL_NAME,
                "prediction_mode":  PREDICTION_MODE,
                "mode_suffix":      MODE_SUFFIX,
                "validation_mode":  info["validation_mode"],
                "focal_year":       info["focal_year"],
                "holdout_iso":      info["holdout_iso"],
                "parameter":        f"spline_alpha_{k}",
                "posterior_mean":   float(np.mean(vals)),
                "posterior_q05":    float(np.percentile(vals, 5)),
                "posterior_q95":    float(np.percentile(vals, 95)),
                "posterior_sd":     float(np.std(vals)),
                "n_train_rows":     info["n_train_rows"],
                "n_train_countries": info["n_train_countries"],
            })

    if "fold_priors" in info:
        lut = {
            "beta_gdp":     "gdp_elasticity",
            "beta_u5m":     "u5_mortality_effect",
            "beta_rural":   "rural_share_effect",
            "beta_edu_fem": "female_education_effect",
            "gamma0_raw":   "baseline_inequality",
        }
        for key, (pm, ps) in info["fold_priors"].items():
            m = [r for r in rows if r["parameter"] == lut.get(key)]
            if m:
                m[0]["prior_mean"] = float(pm)
                m[0]["prior_sd"]   = float(ps)

    df = pd.DataFrame(rows)
    for c in ["prior_mean", "prior_sd"]:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = df[c].astype(float)
    return df


# ============================================================
# BLOCK 7 — METRICS AGGREGATION
# ============================================================

def summarize_metrics(mdf, group_cols):
    """Aggregate fold-level metrics by grouping columns."""
    if mdf is None or len(mdf) == 0:
        return pd.DataFrame()
    return mdf.groupby(group_cols, dropna=False).agg(
        n_folds=("holdout_iso", "count"),
        total_eval_rows=("n_eval_rows", "sum"),
        mean_mae_log=("mae_log", "mean"),
        mean_rmse_log=("rmse_log", "mean"),
        mean_bias_log=("bias_log", "mean"),
        mean_r2_log=("r2_log", "mean"),
        mean_mape_pct_from_log=("mape_pct_from_log", "mean"),
        mean_mae_cons_from_mu=("mae_cons_from_mu", "mean"),
        mean_rmse_cons_from_mu=("rmse_cons_from_mu", "mean"),
        mean_bias_cons_from_mu=("bias_cons_from_mu", "mean"),
        mean_coverage_90_mu_pct=("coverage_90_mu_pct", "mean"),
        mean_coverage_90_y_pct=("coverage_90_y_pct", "mean"),
    ).reset_index()


# ============================================================
# BLOCK 8 — MASTER OUTPUT WRITER
# ============================================================

def write_master_outputs(rpf, rmf, rparf, apf, amf, aparf, elapsed_minutes=None):
    """
    Consolidate all LOCO fold results into master parquet files.
    Called after each focal year so partial results are always
    available even if the run is interrupted.
    """
    sfx  = MODE_SUFFIX
    mn   = MODEL_NAME

    rp   = pd.concat(rpf,   ignore_index=True) if rpf   else pd.DataFrame()
    rm   = pd.concat(rmf,   ignore_index=True) if rmf   else pd.DataFrame()
    rpar = pd.concat(rparf, ignore_index=True) if rparf else pd.DataFrame()
    ap   = pd.concat(apf,   ignore_index=True) if apf   else pd.DataFrame()
    am   = pd.concat(amf,   ignore_index=True) if amf   else pd.DataFrame()
    apar = pd.concat(aparf, ignore_index=True) if aparf else pd.DataFrame()

    if len(rp):
        rp.to_parquet(MASTER_DIR / f"{mn}_rolling_predictions_master_{sfx}.parquet",  index=False)
    if len(rm):
        rm.to_parquet(MASTER_DIR / f"{mn}_rolling_metrics_master_{sfx}.parquet",      index=False)
        for cols, nm in [
            (["validation_mode", "focal_year", "horizon"],  "by_year_horizon"),
            (["validation_mode", "holdout_iso", "horizon"], "by_iso_horizon"),
            (["validation_mode", "horizon"],                "by_horizon"),
            (["validation_mode", "focal_year"],             "by_year"),
            (["validation_mode", "holdout_iso"],            "by_iso"),
            (["validation_mode"],                           "overall"),
        ]:
            summarize_metrics(rm, cols).to_parquet(
                MASTER_DIR / f"{mn}_rolling_metrics_{nm}_{sfx}.parquet", index=False
            )
    if len(rpar):
        rpar.to_parquet(MASTER_DIR / f"{mn}_rolling_params_master_{sfx}.parquet", index=False)

    if len(ap):
        ap.to_parquet(MASTER_DIR / f"{mn}_alltime_predictions_master_{sfx}.parquet", index=False)
    if len(am):
        am.to_parquet(MASTER_DIR / f"{mn}_alltime_metrics_master_{sfx}.parquet", index=False)
        summarize_metrics(am, ["validation_mode", "holdout_iso"]).to_parquet(
            MASTER_DIR / f"{mn}_alltime_metrics_by_iso_{sfx}.parquet", index=False
        )
        summarize_metrics(am, ["validation_mode"]).to_parquet(
            MASTER_DIR / f"{mn}_alltime_metrics_overall_{sfx}.parquet", index=False
        )
    if len(apar):
        apar.to_parquet(MASTER_DIR / f"{mn}_alltime_params_master_{sfx}.parquet", index=False)

    flt = [x for x in [rm, am] if len(x) > 0]
    cmb = pd.concat(flt, ignore_index=True) if flt else pd.DataFrame()
    if len(cmb):
        cmb.to_parquet(MASTER_DIR / f"{mn}_combined_metrics_master_{sfx}.parquet", index=False)
        summarize_metrics(cmb, ["validation_mode"]).to_parquet(
            MASTER_DIR / f"{mn}_combined_metrics_overall_{sfx}.parquet", index=False
        )
        summarize_metrics(cmb, ["validation_mode", "horizon"]).to_parquet(
            MASTER_DIR / f"{mn}_combined_metrics_by_horizon_{sfx}.parquet", index=False
        )

    save_json({
        "prediction_mode":      PREDICTION_MODE,
        "mode_suffix":          MODE_SUFFIX,
        "rolling_pred_rows":    int(len(rp)),
        "rolling_metric_rows":  int(len(rm)),
        "alltime_pred_rows":    int(len(ap)),
        "alltime_metric_rows":  int(len(am)),
        "elapsed_minutes":      elapsed_minutes,
    }, MASTER_DIR / f"{mn}_master_output_summary_{sfx}.json")


# ============================================================
# BLOCK 9 — EMPIRICAL BAYES: WEIGHTED OLS PRIORS
# ============================================================

def fit_empirical_priors(df_train):
    """
    Compute empirical Bayes prior means via weighted OLS.

    A weighted least squares regression of log-consumption on
    the core covariates is run on the training fold.  The
    resulting coefficients become prior means for the Bayesian
    model.  This is fold-specific — no data leakage.  The
    regression includes region-specific slopes on logit(p) to
    capture regional inequality differences.
    """
    cols     = [TARGET, GDP_COL, U5_COL, RURAL_COL, EDU_FEM_COL, "logit_p", "region"]
    has_w    = "survey_weight" in df_train.columns
    use_cols = cols + (["survey_weight"] if has_w else [])
    df       = df_train[use_cols].dropna(subset=cols).copy()

    y     = df[TARGET].values.astype(np.float64)
    n     = len(y)
    gdp   = df[GDP_COL].values
    u5m   = df[U5_COL].values
    rural = df[RURAL_COL].values
    edu   = df[EDU_FEM_COL].values
    lp    = df["logit_p"].values

    is_eca = (df["region"] == REGION_ECA).values.astype(np.float64)
    is_lac = (df["region"] == REGION_LAC).values.astype(np.float64)
    is_ssa = (df["region"] == REGION_SSA).values.astype(np.float64)

    X = np.column_stack([
        np.ones(n), gdp, u5m, rural, edu, lp,
        lp * is_eca, lp * is_lac, lp * is_ssa,
    ])

    if has_w:
        sw   = np.sqrt(df["survey_weight"].values.astype(np.float64))
        beta = np.linalg.lstsq(X * sw[:, None], y * sw, rcond=None)[0]
    else:
        beta = np.linalg.lstsq(X, y, rcond=None)[0]

    return {
        "intercept":   float(beta[0]),
        "beta_gdp":    float(beta[1]),
        "beta_u5m":    float(beta[2]),
        "beta_rural":  float(beta[3]),
        "beta_edu_fem": float(beta[4]),
        "gamma0":      float(beta[5]),
        "n_rows":      n,
    }


# ============================================================
# BLOCK 10 — SURVEY-COUNT WEIGHTING
# ============================================================

def compute_survey_count_weights(df_train):
    """
    Compute 1/sqrt(n_surveys) weights per country, normalised
    to mean 1.  Downweights data-rich countries without
    discarding their information entirely.
    """
    survey_cy  = (
        df_train.dropna(subset=[TARGET])[["iso", "year"]]
        .drop_duplicates()
        .groupby("iso")
        .size()
    )
    iso_counts = df_train["iso"].map(survey_cy).fillna(1).values.astype(np.float32)
    w          = 1.0 / np.sqrt(iso_counts)
    return w / w.mean()


# ============================================================
# BLOCK 11 — TRAIN/TEST SPLITTING & FEATURE ENGINEERING
# ============================================================

def prepare_loco_train_test(df_full, focal_year, holdout_iso, mode="rolling"):
    """
    Prepare training and test sets for one LOCO fold.

    Performs: temporal/geographic splitting, survey-count
    weighting, empirical Bayes priors, weighted covariate
    centering, RBF basis computation, and hierarchical index
    construction.
    """
    # ── Split ─────────────────────────────────────────────────
    if mode == "rolling":
        df_train = df_full[
            (df_full["year"] < focal_year) & (df_full["iso"] != holdout_iso)
        ].copy()
        if PREDICTION_MODE == "nowcast":
            df_test_raw = df_full[
                (df_full["year"] == focal_year) & (df_full["iso"] == holdout_iso)
            ].copy()
        else:
            mty = focal_year + FORECAST_HORIZON
            df_test_raw = df_full[
                (df_full["year"] >= focal_year)
                & (df_full["year"] <= mty)
                & (df_full["iso"] == holdout_iso)
            ].copy()
    elif mode == "all_time":
        df_train    = df_full[df_full["iso"] != holdout_iso].copy()
        df_test_raw = df_full[df_full["iso"] == holdout_iso].copy()
    else:
        raise ValueError(f"Invalid mode: {mode}")

    # ── Drop rows with missing covariates ─────────────────────
    train_covs = [
        TARGET, GDP_COL, U5_COL, RURAL_COL, EDU_FEM_COL,
        GDP_GROWTH_COL, GOV_REV_COL, RES_RENTS_COL,
        "logit_p", "percentile", "iso", "region", "region23",
    ]
    df_train = df_train.dropna(subset=train_covs).copy()
    df_test  = prep_loco_test_grid(df_test_raw)
    test_covs = [
        GDP_COL, U5_COL, RURAL_COL, EDU_FEM_COL,
        GDP_GROWTH_COL, GOV_REV_COL, RES_RENTS_COL,
        "logit_p", "percentile", "iso", "region", "region23",
    ]
    df_test = df_test.dropna(subset=test_covs).copy()

    if not ALLOW_PURE_PREDICTION:
        df_test = df_test[df_test[TARGET].notna()].copy()

    df_test["horizon"] = (
        (df_test["year"] - focal_year).astype(int) if mode == "rolling" else 0
    )

    # ── Minimum data checks ───────────────────────────────────
    n_train_countries = df_train["iso"].nunique()
    if len(df_train) < MIN_TRAIN_ROWS or n_train_countries < MIN_TRAIN_COUNTRIES:
        return None, None, None
    if len(df_test) == 0:
        return None, None, None

    # ── Survey-count weighting ────────────────────────────────
    df_train["survey_weight"] = compute_survey_count_weights(df_train)
    sw = df_train["survey_weight"].values

    # ── Empirical Bayes priors ────────────────────────────────
    emp = fit_empirical_priors(df_train)

    # ── Weighted covariate centering ──────────────────────────
    cov_center_cols = [
        GDP_COL, U5_COL, RURAL_COL, EDU_FEM_COL,
        GDP_GROWTH_COL, GOV_REV_COL, RES_RENTS_COL,
    ]
    cov_means = {}
    for col in cov_center_cols:
        m = float(np.average(df_train[col].values, weights=sw))
        cov_means[col] = m
        df_train[f"{col}_c"] = df_train[col] - m
        df_test[f"{col}_c"]  = df_test[col]  - m

    df_train["gdp_growth_abs_raw"] = df_train[f"{GDP_GROWTH_COL}_c"].abs()
    gam = float(np.average(df_train["gdp_growth_abs_raw"].values, weights=sw))
    cov_means["gdp_growth_abs_raw"] = gam
    df_train["gdp_growth_abs_c"]  = df_train["gdp_growth_abs_raw"] - gam
    df_train["log_gdp_pp_sq_c"]   = df_train[f"{GDP_COL}_c"] ** 2
    df_test["gdp_growth_abs_raw"] = df_test[f"{GDP_GROWTH_COL}_c"].abs()
    df_test["gdp_growth_abs_c"]   = df_test["gdp_growth_abs_raw"] - gam
    df_test["log_gdp_pp_sq_c"]    = df_test[f"{GDP_COL}_c"] ** 2

    # ── RBF basis ─────────────────────────────────────────────
    df_train["_rbf_basis"] = list(compute_rbf_basis(df_train["logit_p"].values))
    df_test["_rbf_basis"]  = list(compute_rbf_basis(df_test["logit_p"].values))

    # ── Hierarchical indices ──────────────────────────────────
    iso_cats    = pd.Categorical(df_train["iso"])
    iso_to_idx  = {v: k for k, v in dict(enumerate(iso_cats.categories)).items()}
    df_train["iso_idx"] = iso_cats.codes

    region_cats  = pd.Categorical(df_train["region"])
    region_to_idx = {v: k for k, v in dict(enumerate(region_cats.categories)).items()}
    df_train["region_idx"] = region_cats.codes

    r23_cats   = pd.Categorical(df_train["region23"])
    r23_to_idx = {v: k for k, v in dict(enumerate(r23_cats.categories)).items()}
    df_train["r23_idx"] = r23_cats.codes

    tmp = (
        df_train[["region23", "region", "r23_idx", "region_idx"]]
        .drop_duplicates().sort_values("r23_idx")
    )
    r23_to_region = (
        tmp.groupby("r23_idx")["region_idx"].first()
        .sort_index().to_numpy().astype(np.int32)
        if len(tmp) > 0 else np.array([], dtype=np.int32)
    )

    df_test["iso_idx"]    = df_test["iso"].map(iso_to_idx).astype("Int64")
    df_test["region_idx"] = df_test["region"].map(region_to_idx).fillna(-1).astype(int)
    df_test["r23_idx"]    = df_test["region23"].map(r23_to_idx).fillna(-1).astype(int)

    # ── Prior construction ────────────────────────────────────
    beta0_cm = (
        emp["intercept"]
        + emp["beta_gdp"]     * cov_means[GDP_COL]
        + emp["beta_u5m"]     * cov_means[U5_COL]
        + emp["beta_rural"]   * cov_means[RURAL_COL]
        + emp["beta_edu_fem"] * cov_means[EDU_FEM_COL]
    )
    g0rpm = inverse_shape_transform(
        np.clip(emp["gamma0"], SHAPE_EPS + 0.01, SHAPE_MAX - 0.01)
    )

    fold_priors = {
        "beta_gdp":     (emp["beta_gdp"],     PRIOR_SD["beta_gdp"]),
        "beta_u5m":     (emp["beta_u5m"],     PRIOR_SD["beta_u5m"]),
        "beta_rural":   (emp["beta_rural"],   PRIOR_SD["beta_rural"]),
        "beta_edu_fem": (emp["beta_edu_fem"], PRIOR_SD["beta_edu_fem"]),
        "gamma0_raw":   (g0rpm,               GAMMA0_RAW_PRIOR_SD),
    }

    info = {
        "model_name":          MODEL_NAME,
        "likelihood":          LIKELIHOOD_NAME,
        "prediction_mode":     PREDICTION_MODE,
        "mode_suffix":         MODE_SUFFIX,
        "validation_mode":     mode,
        "focal_year":          int(focal_year),
        "holdout_iso":         holdout_iso,
        "n_train_rows":        int(len(df_train)),
        "n_test_rows":         int(len(df_test)),
        "n_train_countries":   int(n_train_countries),
        "n_region":            len(region_to_idx),
        "n_r23":               len(r23_to_idx),
        "r23_to_region":       r23_to_region,
        "beta0_prior":         (float(beta0_cm), BETA0_PRIOR_SD),
        "fold_priors":         fold_priors,
        "emp_ols_intercept":   emp["intercept"],
        "emp_ols_gamma0":      emp["gamma0"],
        "emp_ols_n_rows":      emp["n_rows"],
    }

    return df_train, df_test, info


# ============================================================
# BLOCK 12 — PRODUCTION DATA PREPARATION
# ============================================================

def prepare_production_data(df_full, target_years):
    """
    Prepare training and prediction sets for production output.

    Trains on ALL rows with complete data, predicts on every
    LMIC country-year in the target range — including those
    without consumption surveys.

    Prediction target = (all LMICs) ∪ (all countries with
    at least one survey).
    """
    min_year = min(target_years)
    max_year = max(target_years)

    train_covs = [
        TARGET, GDP_COL, U5_COL, RURAL_COL, EDU_FEM_COL,
        GDP_GROWTH_COL, GOV_REV_COL, RES_RENTS_COL,
        "logit_p", "percentile", "iso", "region", "region23",
    ]
    df_train          = df_full.dropna(subset=train_covs).copy()
    n_train_countries = df_train["iso"].nunique()

    if len(df_train) < MIN_TRAIN_ROWS or n_train_countries < MIN_TRAIN_COUNTRIES:
        print("  WARNING: insufficient training data for production model.")
        return None, None, None

    survey_isos = set(df_train["iso"].unique())

    if "income_group" not in df_full.columns:
        print("  WARNING: income_group not found — predicting on all countries.")
        lmic_isos = set(df_full["iso"].unique())
    else:
        lmic_isos = set(
            df_full.loc[df_full["income_group"].isin(LMIC_INCOME_GROUPS), "iso"].unique()
        )
        print(f"  LMIC countries identified: {len(lmic_isos)}")

    target_isos = lmic_isos | survey_isos
    print(f"  Survey countries (incl. HICs): {len(survey_isos)}")
    print(f"  HICs with surveys (added)    : {len(survey_isos - lmic_isos)}")
    print(f"  Total prediction countries   : {len(target_isos)}")

    pred_covs   = [
        GDP_COL, U5_COL, RURAL_COL, EDU_FEM_COL,
        GDP_GROWTH_COL, GOV_REV_COL, RES_RENTS_COL,
        "iso", "region", "region23",
    ]
    df_pred_raw = df_full[
        (df_full["iso"].isin(target_isos))
        & (df_full["year"] >= min_year)
        & (df_full["year"] <= max_year)
    ].dropna(subset=pred_covs).copy()

    if len(df_pred_raw) == 0:
        print("  WARNING: no target country-years with covariates in target range.")
        return None, None, None

    df_predict = prep_loco_test_grid(df_pred_raw)
    df_predict = df_predict.dropna(subset=pred_covs + ["logit_p", "percentile"]).copy()
    df_predict["horizon"] = 0

    # ── Weighting, priors, centering, RBF, indices ────────────
    df_train["survey_weight"] = compute_survey_count_weights(df_train)
    sw  = df_train["survey_weight"].values
    emp = fit_empirical_priors(df_train)

    cov_center_cols = [
        GDP_COL, U5_COL, RURAL_COL, EDU_FEM_COL,
        GDP_GROWTH_COL, GOV_REV_COL, RES_RENTS_COL,
    ]
    cov_means = {}
    for col in cov_center_cols:
        m = float(np.average(df_train[col].values, weights=sw))
        cov_means[col] = m
        df_train[f"{col}_c"]   = df_train[col]   - m
        df_predict[f"{col}_c"] = df_predict[col] - m

    df_train["gdp_growth_abs_raw"]   = df_train[f"{GDP_GROWTH_COL}_c"].abs()
    gam = float(np.average(df_train["gdp_growth_abs_raw"].values, weights=sw))
    cov_means["gdp_growth_abs_raw"]  = gam
    df_train["gdp_growth_abs_c"]     = df_train["gdp_growth_abs_raw"] - gam
    df_train["log_gdp_pp_sq_c"]      = df_train[f"{GDP_COL}_c"] ** 2
    df_predict["gdp_growth_abs_raw"] = df_predict[f"{GDP_GROWTH_COL}_c"].abs()
    df_predict["gdp_growth_abs_c"]   = df_predict["gdp_growth_abs_raw"] - gam
    df_predict["log_gdp_pp_sq_c"]    = df_predict[f"{GDP_COL}_c"] ** 2

    df_train["_rbf_basis"]   = list(compute_rbf_basis(df_train["logit_p"].values))
    df_predict["_rbf_basis"] = list(compute_rbf_basis(df_predict["logit_p"].values))

    iso_cats      = pd.Categorical(df_train["iso"])
    iso_to_idx    = {v: k for k, v in dict(enumerate(iso_cats.categories)).items()}
    df_train["iso_idx"] = iso_cats.codes

    region_cats   = pd.Categorical(df_train["region"])
    region_to_idx = {v: k for k, v in dict(enumerate(region_cats.categories)).items()}
    df_train["region_idx"] = region_cats.codes

    r23_cats      = pd.Categorical(df_train["region23"])
    r23_to_idx    = {v: k for k, v in dict(enumerate(r23_cats.categories)).items()}
    df_train["r23_idx"] = r23_cats.codes

    tmp = (
        df_train[["region23", "region", "r23_idx", "region_idx"]]
        .drop_duplicates().sort_values("r23_idx")
    )
    r23_to_region = (
        tmp.groupby("r23_idx")["region_idx"].first()
        .sort_index().to_numpy().astype(np.int32)
        if len(tmp) > 0 else np.array([], dtype=np.int32)
    )

    df_predict["iso_idx"]    = df_predict["iso"].map(iso_to_idx).astype("Int64")
    df_predict["region_idx"] = df_predict["region"].map(region_to_idx).fillna(-1).astype(int)
    df_predict["r23_idx"]    = df_predict["region23"].map(r23_to_idx).fillna(-1).astype(int)

    beta0_cm = (
        emp["intercept"]
        + emp["beta_gdp"]     * cov_means[GDP_COL]
        + emp["beta_u5m"]     * cov_means[U5_COL]
        + emp["beta_rural"]   * cov_means[RURAL_COL]
        + emp["beta_edu_fem"] * cov_means[EDU_FEM_COL]
    )
    g0rpm = inverse_shape_transform(
        np.clip(emp["gamma0"], SHAPE_EPS + 0.01, SHAPE_MAX - 0.01)
    )
    fold_priors = {
        "beta_gdp":     (emp["beta_gdp"],     PRIOR_SD["beta_gdp"]),
        "beta_u5m":     (emp["beta_u5m"],     PRIOR_SD["beta_u5m"]),
        "beta_rural":   (emp["beta_rural"],   PRIOR_SD["beta_rural"]),
        "beta_edu_fem": (emp["beta_edu_fem"], PRIOR_SD["beta_edu_fem"]),
        "gamma0_raw":   (g0rpm,               GAMMA0_RAW_PRIOR_SD),
    }

    n_survey_rows  = df_predict[TARGET].notna().sum()
    n_pure_rows    = df_predict[TARGET].isna().sum()
    n_pred_ctries  = df_predict["iso"].nunique()
    n_pred_years   = df_predict["year"].nunique()
    print(f"  Training rows    : {len(df_train):,} ({n_train_countries} countries)")
    print(f"  Prediction rows  : {len(df_predict):,} ({n_pred_ctries} countries × {n_pred_years} years)")
    print(f"    with survey    : {int(n_survey_rows):,}")
    print(f"    pure prediction: {int(n_pure_rows):,}")

    info = {
        "model_name":         MODEL_NAME,
        "likelihood":         LIKELIHOOD_NAME,
        "prediction_mode":    PREDICTION_MODE,
        "mode_suffix":        MODE_SUFFIX,
        "validation_mode":    "production",
        "focal_year":         max_year,
        "holdout_iso":        "none",
        "n_train_rows":       int(len(df_train)),
        "n_test_rows":        int(len(df_predict)),
        "n_train_countries":  int(n_train_countries),
        "n_region":           len(region_to_idx),
        "n_r23":              len(r23_to_idx),
        "r23_to_region":      r23_to_region,
        "beta0_prior":        (float(beta0_cm), BETA0_PRIOR_SD),
        "fold_priors":        fold_priors,
        "emp_ols_intercept":  emp["intercept"],
        "emp_ols_gamma0":     emp["gamma0"],
        "emp_ols_n_rows":     emp["n_rows"],
        "lmic_isos":          lmic_isos,
    }

    return df_train, df_predict, info


# ============================================================
# BLOCK 13 — DATAFRAME → JAX ARRAYS (WITH PADDING)
# ============================================================

def _pad_array(arr, target_n, fill_value=0):
    """Pad a JAX array along axis 0 to a target length."""
    n = arr.shape[0]
    if n >= target_n:
        return arr
    pad_n = target_n - n
    pad   = (
        jnp.full(pad_n, fill_value, dtype=arr.dtype)
        if arr.ndim == 1
        else jnp.full((pad_n, arr.shape[1]), fill_value, dtype=arr.dtype)
    )
    return jnp.concatenate([arr, pad])


def df_to_jax(df, include_y=True):
    """
    Convert a training or test DataFrame to padded JAX arrays.

    All arrays are padded to PAD_N.  The obs_mask is True for
    real observations and False for padding — masked entries
    contribute nothing to the ELBO and do not affect inference.
    """
    n_real = len(df)
    rbf    = np.stack(df["_rbf_basis"].values)

    d = {
        "log_gdp":        jnp.array(df[f"{GDP_COL}_c"].values,           dtype=jnp.float32),
        "log_gdp_sq":     jnp.array(df["log_gdp_pp_sq_c"].values,        dtype=jnp.float32),
        "log_u5m":        jnp.array(df[f"{U5_COL}_c"].values,            dtype=jnp.float32),
        "rural":          jnp.array(df[f"{RURAL_COL}_c"].values,         dtype=jnp.float32),
        "edu_fem":        jnp.array(df[f"{EDU_FEM_COL}_c"].values,       dtype=jnp.float32),
        "gdp_growth":     jnp.array(df[f"{GDP_GROWTH_COL}_c"].values,    dtype=jnp.float32),
        "gdp_growth_abs": jnp.array(df["gdp_growth_abs_c"].values,       dtype=jnp.float32),
        "gov_rev":        jnp.array(df[f"{GOV_REV_COL}_c"].values,       dtype=jnp.float32),
        "res_rents":      jnp.array(df[f"{RES_RENTS_COL}_c"].values,     dtype=jnp.float32),
        "pct_basis":      jnp.array(rbf,                                  dtype=jnp.float32),
        "logit_p":        jnp.array(df["logit_p"].values,                dtype=jnp.float32),
        "region_idx":     jnp.array(df["region_idx"].values,             dtype=jnp.int32),
        "r23_idx":        jnp.array(df["r23_idx"].values,                dtype=jnp.int32),
    }

    if include_y and TARGET in df.columns and df[TARGET].notna().all():
        d["y"] = jnp.array(df[TARGET].values, dtype=jnp.float32)
    if "survey_weight" in df.columns:
        d["survey_weight"] = jnp.array(df["survey_weight"].values, dtype=jnp.float32)

    target_n = PAD_N if PAD_N is not None else n_real

    if target_n > n_real:
        obs_mask = jnp.concatenate([
            jnp.ones(n_real,            dtype=jnp.bool_),
            jnp.zeros(target_n - n_real, dtype=jnp.bool_),
        ])
        for key in list(d.keys()):
            fv = 1.0 if key == "survey_weight" else (0 if key in ("region_idx", "r23_idx") else 0.0)
            d[key] = _pad_array(d[key], target_n, fill_value=fv)
    else:
        obs_mask = jnp.ones(n_real, dtype=jnp.bool_)

    d["obs_mask"] = obs_mask
    return d


# ============================================================
# BLOCK 14 — BAYESIAN MODEL
#
# log_cons = LEVEL + SHAPE × logit(p) + SPLINE(logit(p)) + ε
#
# LEVEL  = beta0 + region RE + subregion RE + β·X
# SHAPE  = gamma(r23) × logit(p)   (log-logistic shape)
# SPLINE = Σ_k α_k × RBF_k(logit(p))  + region deviations
# ε      ~ Laplace(0, σ_base + σ_tail × |logit(p)|)
#
# Hierarchy: Global → Region (WB) → Subregion (region23)
# Sum-to-zero constraints at each level ensure identifiability.
# obs_mask excludes padded rows from the likelihood.
# ============================================================

def hierarchical_percentile_model(
    log_gdp, log_gdp_sq, log_u5m, rural, edu_fem,
    gdp_growth, gdp_growth_abs,
    gov_rev, res_rents,
    pct_basis,
    logit_p, region_idx, r23_idx,
    r23_to_region, n_region, n_r23, n_pct_basis,
    beta0_prior, fold_priors,
    y=None, survey_weight=None, obs_mask=None,
):
    """
    Con φ WASE — hierarchical Bayesian consumption distribution model.

    Parameters
    ----------
    log_gdp ... res_rents : JAX arrays (N,)  — centred covariates
    pct_basis             : JAX array  (N, K) — RBF basis evaluations
    logit_p               : JAX array  (N,)  — logit-transformed percentile
    region_idx, r23_idx   : int arrays (N,)  — hierarchy indices
    r23_to_region         : int array  (n_r23,) — subregion → region map
    n_region, n_r23       : int — number of regions / subregions
    n_pct_basis           : int — number of RBF basis functions
    beta0_prior           : (mean, sd) tuple
    fold_priors           : dict of (mean, sd) tuples
    y                     : observed log_cons; None during prediction
    survey_weight         : per-row downweighting; None during prediction
    obs_mask              : bool array; False for padded rows
    """
    N = log_gdp.shape[0]

    # ── LEVEL: average consumption ────────────────────────────
    beta0             = numpyro.sample("beta0",             dist.Normal(*beta0_prior))
    beta_gdp          = numpyro.sample("beta_gdp",          dist.Normal(*fold_priors["beta_gdp"]))
    beta_gdp_sq       = numpyro.sample("beta_gdp_sq",       dist.Normal(*EXTRA_PRIORS["beta_gdp_sq"]))
    beta_u5m          = numpyro.sample("beta_u5m",          dist.Normal(*fold_priors["beta_u5m"]))
    beta_rural        = numpyro.sample("beta_rural",        dist.Normal(*fold_priors["beta_rural"]))
    beta_edu_fem      = numpyro.sample("beta_edu_fem",      dist.Normal(*fold_priors["beta_edu_fem"]))
    beta_gdp_growth   = numpyro.sample("beta_gdp_growth",   dist.Normal(*EXTRA_PRIORS["beta_gdp_growth"]))
    beta_gdp_growth_abs = numpyro.sample("beta_gdp_growth_abs", dist.Normal(*EXTRA_PRIORS["beta_gdp_growth_abs"]))
    beta_gov_rev      = numpyro.sample("beta_gov_rev",      dist.Normal(*EXTRA_PRIORS["beta_gov_rev"]))
    beta_res_rents    = numpyro.sample("beta_res_rents",    dist.Normal(*EXTRA_PRIORS["beta_res_rents"]))

    # Region-level random intercepts (sum-to-zero)
    sigma_region_level = numpyro.sample("sigma_region_level", dist.HalfNormal(SIGMA_REGION_LEVEL_PRIOR))
    with numpyro.plate("region_level_plate", n_region):
        z_region_level_raw = numpyro.sample("z_region_level", dist.Normal(0.0, 1.0))
    z_region_level = z_region_level_raw - z_region_level_raw.mean()
    mu_region      = numpyro.deterministic("mu_region", sigma_region_level * z_region_level)

    # Subregion-level random intercepts (sum-to-zero within region)
    sigma_r23_level = numpyro.sample("sigma_r23_level", dist.HalfNormal(SIGMA_R23_LEVEL_PRIOR))
    with numpyro.plate("r23_level_plate", n_r23):
        z_r23_level = numpyro.sample("z_r23_level", dist.Normal(0.0, 1.0))
    mu_r23_dev    = numpyro.deterministic("mu_r23_dev", sigma_r23_level * z_r23_level)
    region_counts = jnp.bincount(r23_to_region, length=n_region)
    region_sums   = jnp.bincount(r23_to_region, weights=mu_r23_dev, length=n_region)
    region_means  = region_sums / jnp.clip(region_counts, a_min=1)
    mu_r23        = numpyro.deterministic("mu_r23", mu_r23_dev - region_means[r23_to_region])

    level = (
        beta0
        + mu_region[region_idx]
        + mu_r23[r23_idx]
        + beta_gdp          * log_gdp
        + beta_gdp_sq       * log_gdp_sq
        + beta_u5m          * log_u5m
        + beta_rural        * rural
        + beta_edu_fem      * edu_fem
        + beta_gdp_growth   * gdp_growth
        + beta_gdp_growth_abs * gdp_growth_abs
        + beta_gov_rev      * gov_rev
        + beta_res_rents    * res_rents
    )

    # ── SHAPE: inequality (log-logistic gamma) ────────────────
    gamma0_raw = numpyro.sample("gamma0_raw", dist.Normal(*fold_priors["gamma0_raw"]))

    sigma_region_shape = numpyro.sample("sigma_region_shape", dist.HalfNormal(SIGMA_REGION_SHAPE_PRIOR))
    with numpyro.plate("region_shape_plate", n_region):
        z_region_shape_raw = numpyro.sample("z_region_shape", dist.Normal(0.0, 1.0))
    z_region_shape = z_region_shape_raw - z_region_shape_raw.mean()
    s_region       = numpyro.deterministic("s_region", sigma_region_shape * z_region_shape)

    sigma_r23_shape = numpyro.sample("sigma_r23_shape", dist.HalfNormal(SIGMA_R23_SHAPE_PRIOR))
    with numpyro.plate("r23_shape_plate", n_r23):
        z_r23_shape = numpyro.sample("z_r23_shape", dist.Normal(0.0, 1.0))
    s_r23_dev         = numpyro.deterministic("s_r23_dev", sigma_r23_shape * z_r23_shape)
    region_shape_sums = jnp.bincount(r23_to_region, weights=s_r23_dev, length=n_region)
    region_shape_means = region_shape_sums / jnp.clip(region_counts, a_min=1)
    s_r23             = numpyro.deterministic("s_r23", s_r23_dev - region_shape_means[r23_to_region])

    gamma_region = numpyro.deterministic("gamma_region", shape_transform(gamma0_raw + s_region))
    gamma_r23    = numpyro.deterministic("gamma_r23",    shape_transform(gamma0_raw + s_region[r23_to_region] + s_r23))
    gamma0       = numpyro.deterministic("gamma0",       shape_transform(gamma0_raw))

    shape_term = gamma_r23[r23_idx] * logit_p

    # ── SPLINE: nonlinear percentile corrections (RBF) ────────
    with numpyro.plate("spline_plate", n_pct_basis):
        alpha_spline = numpyro.sample("alpha_spline", dist.Normal(0.0, ALPHA_SPLINE_PRIOR_SD))

    sigma_region_spline = numpyro.sample("sigma_region_spline", dist.HalfNormal(SIGMA_REGION_SPLINE_PRIOR))
    with numpyro.plate("region_spline_flat", n_region * n_pct_basis):
        z_region_spline_flat = numpyro.sample("z_region_spline", dist.Normal(0.0, 1.0))
    alpha_region_spline = (sigma_region_spline * z_region_spline_flat).reshape((n_region, n_pct_basis))

    spline_coefs = alpha_spline[None, :] + alpha_region_spline[region_idx, :]
    spline_term  = jnp.sum(pct_basis * spline_coefs, axis=1)

    mu = level + shape_term + spline_term
    numpyro.deterministic("mu_pred", mu)

    # ── Observation noise ─────────────────────────────────────
    sigma_base = numpyro.sample("sigma_base", dist.HalfNormal(0.7))
    sigma_tail = numpyro.sample("sigma_tail", dist.HalfNormal(0.5))
    sigma_obs  = sigma_base + sigma_tail * jnp.abs(logit_p)

    if survey_weight is not None:
        sigma_eff = sigma_obs / jnp.sqrt(jnp.clip(survey_weight, 1e-4, None))
    else:
        sigma_eff = sigma_obs
    numpyro.deterministic("sigma_obs", sigma_obs)

    # ── Likelihood (obs_mask excludes padded rows) ────────────
    with numpyro.plate("obs", N):
        likelihood = dist.Laplace(mu, sigma_eff) if USE_LAPLACE else dist.Normal(mu, sigma_eff)
        if obs_mask is not None:
            likelihood = likelihood.mask(obs_mask)
        numpyro.sample("y_obs", likelihood, obs=y)


# ============================================================
# BLOCK 15 — SVI ENGINE
# ============================================================

def _model_kwargs(data_jax, info, include_y=True):
    kw = dict(
        log_gdp         = data_jax["log_gdp"],
        log_gdp_sq      = data_jax["log_gdp_sq"],
        log_u5m         = data_jax["log_u5m"],
        rural           = data_jax["rural"],
        edu_fem         = data_jax["edu_fem"],
        gdp_growth      = data_jax["gdp_growth"],
        gdp_growth_abs  = data_jax["gdp_growth_abs"],
        gov_rev         = data_jax["gov_rev"],
        res_rents       = data_jax["res_rents"],
        pct_basis       = data_jax["pct_basis"],
        logit_p         = data_jax["logit_p"],
        region_idx      = data_jax["region_idx"],
        r23_idx         = data_jax["r23_idx"],
        r23_to_region   = jnp.array(info["r23_to_region"], dtype=jnp.int32),
        n_region        = info["n_region"],
        n_r23           = info["n_r23"],
        n_pct_basis     = N_RBF_BASIS,
        beta0_prior     = info["beta0_prior"],
        fold_priors     = info["fold_priors"],
        obs_mask        = data_jax.get("obs_mask", None),
    )
    kw["y"]             = data_jax["y"] if (include_y and "y" in data_jax) else None
    kw["survey_weight"] = data_jax.get("survey_weight", None) if include_y else None
    return kw


class SVIWrapper:
    """Lightweight wrapper holding posterior samples and SVI params."""
    def __init__(self, samples, params):
        self.samples = samples
        self.params  = params

    def get_samples(self):
        return self.samples


def fit_svi(data_jax, info, rng_key):
    """
    Fit WASE via SVI with an AutoBNAFNormal guide and a cosine
    decay learning rate schedule (SVI_LR → 1% of SVI_LR over
    SVI_STEPS steps).
    """
    guide     = AutoBNAFNormal(hierarchical_percentile_model, num_flows=1, hidden_factors=[8, 8])
    schedule  = optax.cosine_decay_schedule(init_value=SVI_LR, decay_steps=SVI_STEPS, alpha=0.01)
    optimizer = optax.adam(learning_rate=schedule)
    svi       = SVI(hierarchical_percentile_model, guide, optimizer, loss=Trace_ELBO())

    svi_result = svi.run(
        rng_key, num_steps=SVI_STEPS, progress_bar=False,
        **_model_kwargs(data_jax, info, include_y=True),
    )

    rng_key, kg, km = jrandom.split(rng_key, 3)
    ns = POSTERIOR_SAMPLE_GROUPS * POSTERIOR_SAMPLES

    gp = Predictive(guide, params=svi_result.params, num_samples=ns)
    gs = gp(kg)
    mp = Predictive(hierarchical_percentile_model, posterior_samples=gs)
    ms = mp(km, **_model_kwargs(data_jax, info, include_y=False))

    return SVIWrapper({**gs, **ms}, svi_result.params)


# ============================================================
# BLOCK 16 — PREDICTION
# ============================================================

def predict_test_year(posterior_fit, df_test, info, rng_key):
    """
    Generate WASE predictions for the test set using posterior
    samples.  For countries whose region/subregion was not seen
    in training, falls back to the global mean (level) or
    global gamma (shape).  Observation noise is inflated by
    SVI_VARIANCE_INFLATION for coverage calibration.
    """
    if len(df_test) == 0:
        return pd.DataFrame()

    samples  = posterior_fit.get_samples()
    nd       = samples["beta0"].shape[0]

    beta0    = np.array(samples["beta0"])
    bg       = np.array(samples["beta_gdp"])
    bgsq     = np.array(samples["beta_gdp_sq"])
    bu       = np.array(samples["beta_u5m"])
    br       = np.array(samples["beta_rural"])
    be       = np.array(samples["beta_edu_fem"])
    bgg      = np.array(samples["beta_gdp_growth"])
    bgga     = np.array(samples["beta_gdp_growth_abs"])
    b_gov    = np.array(samples["beta_gov_rev"])
    b_rents  = np.array(samples["beta_res_rents"])
    alpha_spl = np.array(samples["alpha_spline"])

    sb = np.array(samples["sigma_base"]) * SVI_VARIANCE_INFLATION
    st = np.array(samples["sigma_tail"]) * SVI_VARIANCE_INFLATION

    mra   = np.array(samples["mu_region"])
    mr23a = np.array(samples["mu_r23"])
    gra   = np.array(samples["gamma_region"])
    gr23a = np.array(samples["gamma_r23"])
    g0a   = np.array(samples["gamma0"])

    sigma_rs     = np.array(samples["sigma_region_spline"])
    z_rs_flat    = np.array(samples["z_region_spline"])
    n_region     = info["n_region"]
    alpha_reg_spl = (sigma_rs[:, None] * z_rs_flat).reshape(nd, n_region, N_RBF_BASIS)

    # Pad random effects with fallback column for unseen regions
    mrp             = np.concatenate([mra,   np.zeros((nd, 1))], axis=1)
    mr23p           = np.concatenate([mr23a, np.zeros((nd, 1))], axis=1)
    grp             = np.concatenate([gra,   g0a[:, None]],       axis=1)
    gr23p           = np.concatenate([gr23a, np.zeros((nd, 1))], axis=1)
    alpha_reg_spl_p = np.concatenate([alpha_reg_spl, np.zeros((nd, 1, N_RBF_BASIS))], axis=1)

    ri_raw  = df_test["region_idx"].values.copy()
    r23_raw = df_test["r23_idx"].values.copy()
    rs      = ri_raw  != -1
    r23s    = r23_raw != -1
    ri_t    = ri_raw.copy();  ri_t[~rs]   = n_region
    r23_t   = r23_raw.copy(); r23_t[~r23s] = info["n_r23"]

    lg   = df_test[f"{GDP_COL}_c"].values
    lgsq = df_test["log_gdp_pp_sq_c"].values
    lu   = df_test[f"{U5_COL}_c"].values
    ru_t = df_test[f"{RURAL_COL}_c"].values
    ef   = df_test[f"{EDU_FEM_COL}_c"].values
    gt   = df_test[f"{GDP_GROWTH_COL}_c"].values
    gat  = df_test["gdp_growth_abs_c"].values
    gr_t = df_test[f"{GOV_REV_COL}_c"].values
    rr_t = df_test[f"{RES_RENTS_COL}_c"].values
    lpt  = df_test["logit_p"].values
    rbf_test = np.stack(df_test["_rbf_basis"].values)

    level = (
        beta0[:, None]
        + mrp[:, ri_t]
        + mr23p[:, r23_t]
        + bg[:, None]    * lg[None, :]
        + bgsq[:, None]  * lgsq[None, :]
        + bu[:, None]    * lu[None, :]
        + br[:, None]    * ru_t[None, :]
        + be[:, None]    * ef[None, :]
        + bgg[:, None]   * gt[None, :]
        + bgga[:, None]  * gat[None, :]
        + b_gov[:, None] * gr_t[None, :]
        + b_rents[:, None] * rr_t[None, :]
    )

    gamma_eff   = np.where(r23s[None, :], gr23p[:, r23_t], grp[:, ri_t])
    shape       = gamma_eff * lpt[None, :]
    spline_coefs = alpha_spl[:, None, :] + alpha_reg_spl_p[:, ri_t, :]
    spline_term  = np.sum(spline_coefs * rbf_test[None, :, :], axis=2)

    mu_draws    = level + shape + spline_term

    sobs     = sb[:, None] + st[:, None] * np.abs(lpt[None, :])
    rng_key, nk = jrandom.split(rng_key)
    noise    = np.array(
        jrandom.laplace(nk, shape=mu_draws.shape) if USE_LAPLACE
        else jrandom.normal(nk, shape=mu_draws.shape)
    )
    y_draws  = mu_draws + noise * sobs

    dp = df_test.copy()
    dp["model_name"]      = MODEL_NAME
    dp["likelihood"]      = LIKELIHOOD_NAME
    dp["prediction_mode"] = PREDICTION_MODE
    dp["mode_suffix"]     = MODE_SUFFIX
    dp["validation_mode"] = info["validation_mode"]
    dp["focal_year"]      = info["focal_year"]
    dp["holdout_iso"]     = info["holdout_iso"]

    dp["mu_mean"] = mu_draws.mean(0)
    dp["mu_q05"]  = np.percentile(mu_draws, 5,  0)
    dp["mu_q95"]  = np.percentile(mu_draws, 95, 0)
    dp["y_mean"]  = y_draws.mean(0)
    dp["y_q05"]   = np.percentile(y_draws,  5,  0)
    dp["y_q95"]   = np.percentile(y_draws,  95, 0)

    dp["cons_obs_from_log"] = np.where(dp[TARGET].notna(), safe_exp(dp[TARGET]), np.nan)
    dp["cons_mu_mean"]  = safe_exp(dp["mu_mean"])
    dp["cons_mu_q05"]   = safe_exp(dp["mu_q05"])
    dp["cons_mu_q95"]   = safe_exp(dp["mu_q95"])
    dp["cons_y_mean"]   = safe_exp(dp["y_mean"])
    dp["cons_y_q05"]    = safe_exp(dp["y_q05"])
    dp["cons_y_q95"]    = safe_exp(dp["y_q95"])
    dp["unseen_r23"]    = ~r23s
    dp["unseen_region"] = ~rs

    return dp


# ============================================================
# BLOCK 17 — MAIN EXECUTION
# ============================================================

if __name__ == "__main__":
    grand_t0   = time.time()
    master_key = jrandom.PRNGKey(SEED)

    # ── Country list ──────────────────────────────────────────
    master_df = pd.read_parquet(
        INPUT_DIR / f"{INPUT_PREFIX}_{ALL_TIME_VINTAGE}.parquet",
        columns=["iso"],
    )
    ALL_ISOS = sorted(master_df["iso"].unique().tolist())

    # ── Pre-scan: determine PAD_N ─────────────────────────────
    # Scan all vintage files to find the maximum training set
    # size.  Padding every array to this size means JAX
    # compiles the SVI step function exactly once.
    print("Scanning vintage files for padding dimensions...")
    max_rows = 0
    for fy in FOCAL_YEARS:
        vp = INPUT_DIR / f"{INPUT_PREFIX}_{fy}.parquet"
        if vp.exists():
            max_rows = max(max_rows, len(pd.read_parquet(vp, columns=["iso"])))
    for v in set([ALL_TIME_VINTAGE, PRODUCTION_VINTAGE]):
        vp = INPUT_DIR / f"{INPUT_PREFIX}_{v}.parquet"
        if vp.exists():
            max_rows = max(max_rows, len(pd.read_parquet(vp, columns=["iso"])))

    PAD_N = int(np.ceil(max_rows / 1000) * 1000)
    print(f"  Max rows: {max_rows:,}  |  PAD_N: {PAD_N:,}")
    print(f"  → JAX compiles SVI once across all folds.\n")

    run_loco       = RUN_MODE in ("loco", "both")
    run_production = RUN_MODE in ("production", "both")

    print("=" * 80)
    print(f"Con φ {VERSION}  ~  WASE  (Without Any Survey Estimate)")
    print("=" * 80)
    print(f"Model       : {MODEL_NAME}  |  Likelihood: {LIKELIHOOD_NAME}")
    print(f"Input prefix: {INPUT_PREFIX}")
    print(f"Input dir   : {INPUT_DIR}")
    print(f"Output dir  : {WASE_OUT_DIR}")
    print(f"Mode        : {MODE_SUFFIX}")
    print(f"Focal years : {FOCAL_YEARS[0]}–{FOCAL_YEARS[-1]}")
    print(f"Run mode    : {RUN_MODE}")
    if run_production:
        print(f"Production  : vintage {PRODUCTION_VINTAGE}, "
              f"years {PRODUCTION_YEARS[0]}–{PRODUCTION_YEARS[-1]}")
    print("=" * 80)
    save_json(build_run_metadata(), MASTER_DIR / f"{MODEL_NAME}_run_metadata_{MODE_SUFFIX}.json")

    rpf, rmf, rparf = [], [], []
    apf, amf, aparf = [], [], []
    fold_counter    = 0

    PRED_COLS = [
        "model_name", "likelihood", "prediction_mode", "mode_suffix",
        "validation_mode", "focal_year", "holdout_iso", "horizon",
        "iso", "year", "percentile", "region", "region23",
        TARGET, "cons", "cons_obs_from_log",
        "mu_mean", "mu_q05", "mu_q95", "y_mean", "y_q05", "y_q95",
        "cons_mu_mean", "cons_mu_q05", "cons_mu_q95",
        "cons_y_mean", "cons_y_q05", "cons_y_q95",
        "unseen_r23", "unseen_region",
    ]

    # ══════════════════════════════════════════════════════════
    # PHASE 1: ROLLING LOCO
    # For each focal year, train on all data before that year,
    # hold out one country at a time and predict its distribution.
    # ══════════════════════════════════════════════════════════
    if not run_loco:
        print(f"\n{'=' * 80}")
        print(f"PHASE 1: ROLLING LOCO — SKIPPED (RUN_MODE = {RUN_MODE})")
        print(f"{'=' * 80}")
    else:
        print(f"\n{'=' * 80}")
        print(f"PHASE 1: ROLLING LOCO  ({MODE_SUFFIX.upper()})")
        print(f"{'=' * 80}")

        for fy in FOCAL_YEARS:
            yd = ROLLING_DIR / str(fy)
            yd.mkdir(exist_ok=True)

            vp = INPUT_DIR / f"{INPUT_PREFIX}_{fy}.parquet"
            if not vp.exists():
                print(f"  Skip {fy}: no feature file")
                continue

            print(f"\n{'-' * 80}\nFOCAL YEAR {fy}\n{'-' * 80}")
            dff = pd.read_parquet(vp)

            for iso in ALL_ISOS:
                sp = yd / f"{MODEL_NAME}_pred_{fy}_{iso}_{MODE_SUFFIX}.parquet"
                mp = yd / f"{MODEL_NAME}_metrics_{fy}_{iso}_{MODE_SUFFIX}.parquet"
                if sp.exists() and mp.exists():
                    continue
                if not (dff["iso"] == iso).any():
                    continue

                master_key, sk, pk = jrandom.split(master_key, 3)
                dtr, dte, inf = prepare_loco_train_test(dff, fy, iso, "rolling")
                if dtr is None or dte is None or len(dte) == 0:
                    continue

                t0  = time.time()
                djt = df_to_jax(dtr, True)
                pf  = fit_svi(djt, inf, sk)
                dp  = predict_test_year(pf, dte, inf, pk)

                fmf = []
                for h in sorted(dp["horizon"].unique()):
                    dh = dp[dp["horizon"] == h].copy()
                    fmf.append(evaluate_predictions(dh, "rolling", fy, iso, int(h)))
                mdf = pd.concat(fmf, ignore_index=True)
                pdf = summarize_posterior(pf, inf)

                if len(dp) > 0:
                    cts = [c for c in PRED_COLS if c in dp.columns]
                    dp[cts].to_parquet(sp, index=False)
                    rpf.append(dp[cts].copy())
                mdf.to_parquet(mp, index=False)
                rmf.append(mdf.copy())
                rparf.append(pdf.copy())

                el = time.time() - t0
                for _, mr in mdf.iterrows():
                    hh = int(mr["horizon"])
                    if pd.notna(mr["r2_log"]):
                        print(f"[ROLLING] y={fy} iso={iso} h={hh} {el:.1f}s R²={mr['r2_log']:.4f}")
                    else:
                        print(f"[ROLLING] y={fy} iso={iso} h={hh} {el:.1f}s no eval")

                del pf, djt, dtr, dte, dp, mdf, pdf
                gc.collect()
                fold_counter += 1
                if fold_counter % JAX_CLEAR_EVERY == 0:
                    jax.clear_caches()

            write_master_outputs(rpf, rmf, rparf, apf, amf, aparf,
                                 round((time.time() - grand_t0) / 60, 2))
            print(f"Saved rolling master through {fy}.")

    # ══════════════════════════════════════════════════════════
    # PHASE 2: ALL-TIME LOCO
    # Train on all years, hold out one country at a time.
    # ══════════════════════════════════════════════════════════
    if not run_loco:
        print(f"\n{'=' * 80}")
        print(f"PHASE 2: ALL-TIME LOCO — SKIPPED (RUN_MODE = {RUN_MODE})")
        print(f"{'=' * 80}")
    else:
        print(f"\n{'=' * 80}")
        print(f"PHASE 2: ALL-TIME LOCO  (VINTAGE {ALL_TIME_VINTAGE})")
        print(f"{'=' * 80}")

        dat = pd.read_parquet(INPUT_DIR / f"{INPUT_PREFIX}_{ALL_TIME_VINTAGE}.parquet")

        for iso in ALL_ISOS:
            sp = ALL_TIME_DIR / f"{MODEL_NAME}_pred_alltime_{iso}_{MODE_SUFFIX}.parquet"
            mp = ALL_TIME_DIR / f"{MODEL_NAME}_metrics_alltime_{iso}_{MODE_SUFFIX}.parquet"
            if sp.exists() and mp.exists():
                continue
            if not (dat["iso"] == iso).any():
                continue

            master_key, sk, pk = jrandom.split(master_key, 3)
            dtr, dte, inf = prepare_loco_train_test(dat, ALL_TIME_VINTAGE, iso, "all_time")
            if dtr is None or dte is None or len(dte) == 0:
                continue

            t0  = time.time()
            djt = df_to_jax(dtr, True)
            pf  = fit_svi(djt, inf, sk)
            dp  = predict_test_year(pf, dte, inf, pk)
            mdf = evaluate_predictions(dp, "all_time", ALL_TIME_VINTAGE, iso, 0)
            pdf = summarize_posterior(pf, inf)

            if len(dp) > 0:
                cts = [c for c in PRED_COLS if c in dp.columns]
                dp[cts].to_parquet(sp, index=False)
                apf.append(dp[cts].copy())
            mdf.to_parquet(mp, index=False)
            amf.append(mdf.copy())
            aparf.append(pdf.copy())

            el = time.time() - t0
            if pd.notna(mdf["r2_log"].iloc[0]):
                print(f"[ALL-TIME] iso={iso} {el:.1f}s "
                      f"n_eval={int(mdf['n_eval_rows'].iloc[0])} "
                      f"R²={mdf['r2_log'].iloc[0]:.4f}")
            else:
                print(f"[ALL-TIME] iso={iso} {el:.1f}s no eval")

            del pf, djt, dtr, dte, dp, mdf, pdf
            gc.collect()
            fold_counter += 1
            if fold_counter % JAX_CLEAR_EVERY == 0:
                jax.clear_caches()

    # ══════════════════════════════════════════════════════════
    # PHASE 3: PRODUCTION PREDICTIONS
    # Train on all data, predict every LMIC country-year in
    # PRODUCTION_YEARS whether or not a survey exists.
    # ══════════════════════════════════════════════════════════
    if not run_production:
        print(f"\n{'=' * 80}")
        print(f"PHASE 3: PRODUCTION — SKIPPED (RUN_MODE = {RUN_MODE})")
        print(f"{'=' * 80}")
    else:
        print(f"\n{'=' * 80}")
        print(f"PHASE 3: PRODUCTION PREDICTIONS "
              f"(VINTAGE {PRODUCTION_VINTAGE}, "
              f"{PRODUCTION_YEARS[0]}–{PRODUCTION_YEARS[-1]})")
        print(f"{'=' * 80}")

        prod_vp = INPUT_DIR / f"{INPUT_PREFIX}_{PRODUCTION_VINTAGE}.parquet"
        if not prod_vp.exists():
            print(f"  WARNING: feature file not found: {prod_vp}")
            print("  Skipping production predictions.")
        else:
            prod_df = pd.read_parquet(prod_vp)
            print(f"  Feature file : {prod_vp.name}")
            print(f"  Raw rows     : {len(prod_df):,}")
            print(f"  Countries    : {prod_df['iso'].nunique()}")
            print(f"  Years        : {int(prod_df['year'].min())}–{int(prod_df['year'].max())}\n")

            prod_train, prod_predict, prod_info = prepare_production_data(
                prod_df, PRODUCTION_YEARS
            )

            if prod_train is not None and prod_predict is not None:
                master_key, prod_sk, prod_pk = jrandom.split(master_key, 3)

                t0 = time.time()
                print(f"\n  Fitting production model (SVI, {SVI_STEPS:,} steps)...")
                prod_djt = df_to_jax(prod_train, True)
                prod_fit = fit_svi(prod_djt, prod_info, prod_sk)
                print(f"  SVI fit complete: {time.time() - t0:.1f}s")

                print(f"\n  Generating predictions...")
                pred_isos    = sorted(prod_predict["iso"].unique())
                prod_chunks  = []

                for pi, piso in enumerate(pred_isos):
                    df_iso = prod_predict[prod_predict["iso"] == piso].copy()
                    if len(df_iso) == 0:
                        continue
                    master_key, chunk_pk = jrandom.split(master_key)
                    dp_chunk = predict_test_year(prod_fit, df_iso, prod_info, chunk_pk)
                    if len(dp_chunk) > 0:
                        prod_chunks.append(dp_chunk)
                    if (pi + 1) % 25 == 0 or (pi + 1) == len(pred_isos):
                        print(f"    {pi + 1}/{len(pred_isos)} countries...")

                if prod_chunks:
                    prod_all = pd.concat(prod_chunks, ignore_index=True)

                    prod_cols = [
                        "model_name", "likelihood", "prediction_mode", "mode_suffix",
                        "validation_mode", "iso", "year", "percentile",
                        "region", "region23", "income_group",
                        TARGET, "cons", "cons_obs_from_log",
                        "mu_mean", "mu_q05", "mu_q95",
                        "y_mean", "y_q05", "y_q95",
                        "cons_mu_mean", "cons_mu_q05", "cons_mu_q95",
                        "cons_y_mean", "cons_y_q05", "cons_y_q95",
                        "unseen_r23", "unseen_region",
                    ]
                    prod_cols  = [c for c in prod_cols if c in prod_all.columns]
                    prod_all["has_survey"] = prod_all[TARGET].notna()
                    prod_all["is_lmic"]    = prod_all["iso"].isin(prod_info.get("lmic_isos", set()))
                    for c in ["has_survey", "is_lmic"]:
                        if c not in prod_cols:
                            prod_cols.append(c)

                    prod_out = PRODUCTION_DIR / f"{MODEL_NAME}_production_predictions_{MODE_SUFFIX}.parquet"
                    prod_all[prod_cols].to_parquet(prod_out, index=False)

                    prod_params = summarize_posterior(prod_fit, prod_info)
                    prod_params.to_parquet(
                        PRODUCTION_DIR / f"{MODEL_NAME}_production_params_{MODE_SUFFIX}.parquet",
                        index=False,
                    )

                    n_total    = len(prod_all)
                    n_survey   = int(prod_all["has_survey"].sum())
                    n_pure     = n_total - n_survey
                    n_countries = prod_all["iso"].nunique()
                    n_lmic     = prod_all.loc[prod_all["is_lmic"], "iso"].nunique()
                    total_time = time.time() - t0

                    print(f"\n  Production predictions complete:")
                    print(f"    Total rows   : {n_total:,}")
                    print(f"    Countries    : {n_countries} ({n_lmic} LMIC + {n_countries - n_lmic} HIC)")
                    print(f"    Years        : {prod_all['year'].nunique()} "
                          f"({int(prod_all['year'].min())}–{int(prod_all['year'].max())})")
                    print(f"    Survey rows  : {n_survey:,}")
                    print(f"    Pure preds   : {n_pure:,}")
                    print(f"    Time         : {total_time:.1f}s")
                    print(f"    Saved to     : {prod_out}")

                    prod_eval = prod_all.dropna(subset=[TARGET])
                    if len(prod_eval) > 0:
                        _y  = prod_eval[TARGET].values
                        _yh = prod_eval["mu_mean"].values
                        _r  = _yh - _y
                        _r2 = 1.0 - np.sum(_r ** 2) / np.sum((_y - _y.mean()) ** 2)
                        print(f"    In-sample R² (survey rows): {_r2:.4f}")

                    save_json({
                        "vintage":            PRODUCTION_VINTAGE,
                        "target_years":       PRODUCTION_YEARS,
                        "n_predictions":      n_total,
                        "n_countries":        n_countries,
                        "n_lmic":             n_lmic,
                        "n_hic":              n_countries - n_lmic,
                        "n_years":            prod_all["year"].nunique(),
                        "n_with_survey":      n_survey,
                        "n_pure_prediction":  n_pure,
                        "total_time_seconds": round(total_time, 1),
                    }, PRODUCTION_DIR / f"{MODEL_NAME}_production_metadata_{MODE_SUFFIX}.json")

                else:
                    print("  WARNING: no production predictions generated.")

                del prod_fit, prod_djt, prod_train, prod_predict
                gc.collect()
                jax.clear_caches()

    # ══════════════════════════════════════════════════════════
    # FINAL MASTER OUTPUTS
    # ══════════════════════════════════════════════════════════
    print(f"\n{'=' * 80}\nWRITING FINAL MASTER OUTPUTS\n{'=' * 80}")
    write_master_outputs(
        rpf, rmf, rparf, apf, amf, aparf,
        round((time.time() - grand_t0) / 60, 2),
    )
    print(f"\nCon φ {VERSION}  WASE  complete.")
    print(f"Mode   : {PREDICTION_MODE}"
          + (f"  |  horizon: {FORECAST_HORIZON} yr" if PREDICTION_MODE == "forecast" else ""))
    print(f"Outputs: {WASE_OUT_DIR}")
    print(f"Runtime: {(time.time() - grand_t0) / 60:.1f} minutes")

    # ══════════════════════════════════════════════════════════
    # PERFORMANCE DIAGNOSTICS
    # ══════════════════════════════════════════════════════════
    print(f"\n{'=' * 80}\nPERFORMANCE DIAGNOSTICS (ROLLING FOLDS)\n{'=' * 80}\n")

    _pred_path = MASTER_DIR / f"{MODEL_NAME}_rolling_predictions_master_{MODE_SUFFIX}.parquet"
    if not _pred_path.exists():
        print("No rolling predictions master — skipping diagnostics.")
    else:
        _dp = pd.read_parquet(_pred_path)
        _dp = _dp[_dp[TARGET].notna()].copy()
        print(f"Eval rows: {len(_dp):,}  |  Countries: {_dp['iso'].nunique()}")
        print(f"Focal years: {sorted(_dp['focal_year'].unique().tolist())}\n")

        def _compute_metrics(df):
            y   = df[TARGET].values
            yh  = df["mu_mean"].values
            r   = yh - y
            ss_res = np.sum(r ** 2)
            ss_tot = np.sum((y - y.mean()) ** 2)
            cov_mu = np.mean((y >= df["mu_q05"].values) & (y <= df["mu_q95"].values)) * 100
            cov_y  = np.mean((y >= df["y_q05"].values)  & (y <= df["y_q95"].values))  * 100
            return {
                "n_obs":          len(df),
                "n_countries":    df["iso"].nunique(),
                "mae_log":        np.mean(np.abs(r)),
                "rmse_log":       np.sqrt(np.mean(r ** 2)),
                "bias_log":       r.mean(),
                "r2_log":         1.0 - ss_res / ss_tot if ss_tot > 1e-8 else np.nan,
                "mape_pct":       np.mean(np.abs(np.exp(np.clip(r, -20, 20)) - 1.0)) * 100,
                "coverage_90_mu": cov_mu,
                "coverage_90_y":  cov_y,
            }

        _overall = pd.DataFrame([_compute_metrics(_dp)])
        print("─" * 60 + "\nOVERALL\n" + "─" * 60)
        for c in _overall.columns:
            v = _overall[c].iloc[0]
            print(f"  {c:25s}  {int(v):>12,}" if c in ("n_obs", "n_countries") else f"  {c:25s}  {v:>12.4f}")
        print()

        _dp["pct_bin"] = pd.cut(
            _dp["percentile"],
            bins=[0, 5, 10, 25, 50, 75, 90, 95, 100],
            labels=["1-5", "6-10", "11-25", "26-50", "51-75", "76-90", "91-95", "96-99"],
        )
        _by_pct = (
            _dp.groupby("pct_bin", observed=True)
            .apply(_compute_metrics).apply(pd.Series).reset_index()
        )
        print("─" * 60 + "\nBY PERCENTILE BIN\n" + "─" * 60)
        print(_by_pct[["pct_bin", "n_obs", "r2_log", "mae_log", "rmse_log",
                        "coverage_90_mu", "coverage_90_y"]].to_string(index=False, float_format="%.4f"))
        print()

        _by_reg = (
            _dp.groupby("region")
            .apply(_compute_metrics).apply(pd.Series).reset_index()
        )
        print("─" * 60 + "\nBY REGION\n" + "─" * 60)
        print(_by_reg[["region", "n_obs", "n_countries", "r2_log", "mae_log",
                        "bias_log", "coverage_90_mu"]].to_string(index=False, float_format="%.4f"))
        print()

        _by_iso = (
            _dp.groupby("iso")
            .apply(_compute_metrics).apply(pd.Series).reset_index()
            .sort_values("r2_log", ascending=True)
        )
        print("─" * 60 + "\nWORST 15 COUNTRIES\n" + "─" * 60)
        print(_by_iso.head(15)[["iso", "n_obs", "r2_log", "mae_log", "bias_log",
                                 "coverage_90_mu"]].to_string(index=False, float_format="%.4f"))
        print(f"\n  Median R²: {_by_iso['r2_log'].median():.4f}  |  Mean: {_by_iso['r2_log'].mean():.4f}")
        print(f"  R²<0: {(_by_iso['r2_log'] < 0).sum()}  |  R²>0.9: {(_by_iso['r2_log'] > 0.9).sum()}")

        _diag_dir = MASTER_DIR / "diagnostics"
        _diag_dir.mkdir(exist_ok=True)
        _overall.to_parquet(_diag_dir / f"{MODEL_NAME}_diag_overall_{MODE_SUFFIX}.parquet",       index=False)
        _by_pct.to_parquet( _diag_dir / f"{MODEL_NAME}_diag_by_percentile_{MODE_SUFFIX}.parquet", index=False)
        _by_reg.to_parquet( _diag_dir / f"{MODEL_NAME}_diag_by_region_{MODE_SUFFIX}.parquet",     index=False)
        _by_iso.to_parquet( _diag_dir / f"{MODEL_NAME}_diag_by_country_{MODE_SUFFIX}.parquet",    index=False)
        print(f"\nDiagnostics saved to: {_diag_dir}")

        del _dp, _overall, _by_pct, _by_reg, _by_iso
        gc.collect()